# ML-10 — Content Action Playbook

This notebook turns the validated refresh-scoring work into a practical, human-reviewed action playbook. It is decision support, not a production automation system.

Lane: Refresh / Content Opportunity Scoring. The baseline rule from Week 4 prioritizes pages with at least 300 impressions and average position 4–20, scoring them by impressions. The Week 6 validation audit uses time-aware validation and explicitly avoids future-window and label-derived inputs.

## 1. Ranked actions + reason codes

**Primary action:** `review_refresh` — send the highest-scoring pages to an editor for review.

**Reason code:** `visible_position_opportunity` — the page has meaningful search volume (>=300 impressions) while sitting in positions 4–20, so it is visible but not yet in the strongest positions.

**Fallback action:** `monitor` — do not prioritize the page in the refresh queue when the rule is not met.

**Ranking rule:** among qualifying pages, rank by current-window impressions descending. This is a transparent hand rule, not a fitted weight.

**Archetype → action mapping**
- High impressions + position 4–20 → `review_refresh` / `visible_position_opportunity`.
- Lower impressions or position outside 4–20 → `monitor` / `insufficient_signal`.
- Strong top-3 visibility → do not automatically recommend a refresh; keep for human review only if another business/editorial reason exists.

The paper's freshness evidence is directionally consistent with reviewing older content for possible refresh, but it does not prove that every old page should be refreshed or that refresh causes the observed lift.

In [ ]:
# Executable queue construction from the March 2026 partition.
# Uses only decision-time fields; no trend labels or future-window outcomes.
!pip -q install duckdb pyarrow
import duckdb, pandas as pd, json, os

rel = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
con = duckdb.connect()
q = f'''SELECT client_hash_id AS client_id, content_hash_id AS content_id,
SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
AVG(gsc_avg_position) AS avg_position,
BOOL_OR(ga4_data_available IS TRUE) AS ga4_available
FROM {rel} GROUP BY 1,2'''
df = con.execute(q).df()
df['ctr'] = df['clicks'] / df['impressions'].replace(0, pd.NA)
df['qualifies'] = (df['impressions'] >= 300) & df['avg_position'].between(4,20, inclusive='both')
df['score'] = df['impressions'].where(df['qualifies'], 0)
df['reason_code'] = df['qualifies'].map({True:'visible_position_opportunity', False:'insufficient_signal'})
df['action'] = df['qualifies'].map({True:'review_refresh', False:'monitor'})
queue = df.sort_values(['score','impressions'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', range(1, len(queue)+1))
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Rows:', len(queue))
print('review_refresh:', int((queue.action=='review_refresh').sum()))
print(queue.head(10)[['rank','client_id','content_id','score','avg_position','reason_code','action']].to_string(index=False))

## 2. Intended use and limits

**Intended user:** an SEO/content editor who needs a short review queue.

**Intended use:** rank pages for human review so limited editorial time is spent first on pages with a measurable visibility opportunity. The output is a prioritization aid, not an automatic publishing decision.

**Limits:** the score is based on the current measurement window; it does not predict Google's algorithm, guarantee traffic growth, or establish that refreshing a page will cause an improvement. Search demand, intent, SERP competition, seasonality, technical issues, business value, and editorial quality are not fully represented. The label/proxy from earlier modeling is an observed outcome, not a causal treatment effect.

**Cost/value thinking:** the main cost is editor time spent reviewing or refreshing a page. A useful queue should therefore favor high-signal pages near valuable positions and avoid spending time on low-volume/noisy pages. The value is potential prioritization efficiency; actual refresh ROI must be measured separately after human decisions and business outcomes are available.

In [ ]:
# Small audit numbers to make the playbook traceable.
print('Top-50 review queue size:', min(50, len(queue)))
print('Top-50 qualifying share:', round((queue.head(50).action=='review_refresh').mean(), 3))
print('Median score among qualifying pages:', queue.loc[queue.qualifies, 'score'].median())
print('Decision rule: impressions >= 300 AND avg_position between 4 and 20')

## 3. Human review + the no-go list

Before any refresh action, a person should check: search intent match, page quality and accuracy, business relevance, current SERP context, cannibalization, technical/indexing issues, and whether the page has already been recently updated. The reviewer can reject, defer, or re-prioritize any recommendation.

**Do NOT automate:** publishing edits; deleting or redirecting pages; changing canonical/indexing directives; changing internal-link architecture at scale; making claims about causality; or applying a refresh solely because a score is high. These actions can have irreversible or high-cost consequences and require human judgment.

The playbook should remain a recommendation queue with an explicit human gate.

In [ ]:
# Human-review checklist for the first 10 ranked recommendations.
review = queue.head(10)[['rank','client_id','content_id','score','avg_position','reason_code','action']].copy()
review['human_checks'] = 'intent; quality; business value; SERP context; cannibalization; technical/indexing status'
review['wrong_if'] = 'data is stale, intent changed, technical issue explains performance, or page has already been refreshed'
print(review.to_string(index=False))

## 4. Monitoring / retrain triggers

Monitor the queue at least monthly or when the underlying search environment changes materially. Suggested triggers: (1) Precision@K on a labeled validation slice drops materially versus the frozen baseline/model benchmark; (2) the distribution of impressions or average position shifts materially; (3) missingness or availability flags change; (4) a new data release changes the field definitions or time coverage; (5) editors repeatedly reject the same reason code.

**Retrain/recalibrate:** only after a fresh time-aware validation split is available and the new version beats the frozen baseline on the same metric and comparable slice. Do not retrain merely because a calendar month passed.

**Light monitoring:** keep the rule frozen, record queue size, top-K review outcomes, rejection rate, and Precision@K when a valid observed outcome becomes available. Investigate drift before changing the rule.

In [ ]:
metrics = {
  'rows_scored': int(len(queue)),
  'review_refresh_rows': int((queue.action=='review_refresh').sum()),
  'top50_review_share': float((queue.head(50).action=='review_refresh').mean()),
  'rule': 'impressions >= 300 AND avg_position between 4 and 20',
  'score': 'impressions for qualifying pages, else 0',
  'decision_status': 'human_review_required'
}
with open('work/outputs/w07_action_playbook_metrics.json','w') as f: json.dump(metrics,f,indent=2)
print(json.dumps(metrics, indent=2))

## 5. Exports for the paper

The queue CSV is intentionally regenerated by the notebook and is not treated as a committed data artifact. The committed metrics JSON is the traceable receipt for the paper. A reusable figure can be added later under `work/figures/` if needed.

The paper should describe the playbook as decision support and preserve the distinction between observed associations and causal claims about refreshing content.

In [ ]:
print('Exported:', 'work/outputs/baseline_action_score.csv')
print('Exported:', 'work/outputs/w07_action_playbook_metrics.json')
assert os.path.exists('work/outputs/baseline_action_score.csv')
assert os.path.exists('work/outputs/w07_action_playbook_metrics.json')

## Self-check

- [x] Ranked actions and reason codes are explicit.
- [x] Archetype → action mapping is explicit.
- [x] Decay/refresh insight is framed as directional/observational, not causal.
- [x] Intended use and limits are stated.
- [x] Human review rules and no-go automation cases are stated.
- [x] Cost/value thinking is included.
- [x] Monitoring and retrain triggers are defined.
- [x] Queue and metrics are exported to `work/outputs/`.
- [ ] Run Runtime → Run all in Colab and save the executed notebook back to GitHub.

Final wording: this is a practical, non-production content prioritization playbook for human review.